In [1]:
# generate read counts

In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
import pickle
from Bio.Seq import Seq
import time
import os
import seaborn as sns
import matplotlib as plt

# scripts
#from timepoint_calculations import timepoint_calculations
from timepoint_calculations_v2 import timepoint_calculations
from add_pkr_metadata import pkr_metadata

In [2]:
date = datetime.now().strftime("%y%m%d")
print(date)

250204


In [10]:
# map variants: PKR - 220823
pkr_variant_table = '../../data/alignparse/pkr.codon_variant_table.csv'
var_df = pd.read_csv(pkr_variant_table)
var_df = var_df.query('n_aa_substitutions < 2')
var_df.loc[var_df.n_aa_substitutions == 0, "aa_substitutions"] = "WT"
var_df.aa_substitutions = 'PKR-' + var_df.aa_substitutions
pkr_dict = dict(zip(var_df.barcode, var_df.aa_substitutions))

In [16]:
# get the sample names
fastq_dir = '../../data/fastq/'
sample_list = ["_".join(s.split("_",4)[:4]) for s in os.listdir(fastq_dir) if os.path.isfile(os.path.join(fastq_dir, s))]
samples = list(set(sample_list))
samples.remove('._Reports')
samples.sort()
samples

# Custom sorting function that extracts the numeric part from each string
def custom_sort(val):
    # Extract the numeric part from the string (assuming it starts with 'file')
    num_part = int(''.join(filter(str.isdigit, val)))
    # Ensure that "10" is placed directly after "9"
    if num_part == 10:
        return 9.5  # Place 10 right after 9
    return num_part

# Sort the list using the custom key
sorted_list = sorted(my_list, key=custom_sort)
print(sorted_list)

['1-0_blank-r1_0_S1',
 '1-12_blank-r1_12_S13',
 '1-16_blank-r1_16_S25',
 '1-20_blank-r1_20_S37',
 '10-0_myxv-r2_0_S10',
 '10-12_myxv-r2_12_S22',
 '10-16_myxv-r2_16_S34',
 '10-20_myxv-r2_20_S46',
 '11-0_tpv-r1_0_S11',
 '11-12_tpv-r1_12_S23',
 '11-16_tpv-r1_16_S35',
 '11-20_tpv-r1_20_S47',
 '12-0_tpv-r2_0_S12',
 '12-12_tpv-r2_12_S24',
 '12-16_tpv-r2_16_S36',
 '12-20_tpv-r2_20_S48',
 '2-0_blank-r2_0_S2',
 '2-12_blank-r2_12_S14',
 '2-16_blank-r2_16_S26',
 '2-20_blank-r2_20_S38',
 '3-0_rcvz-r1_0_S3',
 '3-12_rcvz-r1_12_S15',
 '3-16_rcvz-r1_16_S27',
 '3-20_rcvz-r1_20_S39',
 '4-0_rcvz-r2_0_S4',
 '4-12_rcvz-r2_12_S16',
 '4-16_rcvz-r2_16_S28',
 '4-20_rcvz-r2_20_S40',
 '5-0_vacv-r1_0_S5',
 '5-12_vacv-r1_12_S17',
 '5-16_vacv-r1_16_S29',
 '5-20_vacv-r1_20_S41',
 '6-0_vacv-r2_0_S6',
 '6-12_vacv-r2_12_S18',
 '6-16_vacv-r2_16_S30',
 '6-20_vacv-r2_20_S42',
 '7-0_varv-r1_0_S7',
 '7-12_varv-r1_12_S19',
 '7-16_varv-r1_16_S31',
 '7-20_varv-r1_20_S43',
 '8-0_varv-r2_0_S8',
 '8-12_varv-r2_12_S20',
 '8-16_var

In [17]:
samples = [
 '1-0_blank-r1_0_S1',
 '1-12_blank-r1_12_S13',
 '1-16_blank-r1_16_S25',
 '1-20_blank-r1_20_S37',
 '2-0_blank-r2_0_S2',
 '2-12_blank-r2_12_S14',
 '2-16_blank-r2_16_S26',
 '2-20_blank-r2_20_S38',
 '3-0_rcvz-r1_0_S3',
 '3-12_rcvz-r1_12_S15',
 '3-16_rcvz-r1_16_S27',
 '3-20_rcvz-r1_20_S39',
 '4-0_rcvz-r2_0_S4',
 '4-12_rcvz-r2_12_S16',
 '4-16_rcvz-r2_16_S28',
 '4-20_rcvz-r2_20_S40',
 '5-0_vacv-r1_0_S5',
 '5-12_vacv-r1_12_S17',
 '5-16_vacv-r1_16_S29',
 '5-20_vacv-r1_20_S41',
 '6-0_vacv-r2_0_S6',
 '6-12_vacv-r2_12_S18',
 '6-16_vacv-r2_16_S30',
 '6-20_vacv-r2_20_S42',
 '7-0_varv-r1_0_S7',
 '7-12_varv-r1_12_S19',
 '7-16_varv-r1_16_S31',
 '7-20_varv-r1_20_S43',
 '8-0_varv-r2_0_S8',
 '8-12_varv-r2_12_S20',
 '8-16_varv-r2_16_S32',
 '8-20_varv-r2_20_S44',
 '9-0_myxv-r1_0_S9',
 '9-12_myxv-r1_12_S21',
 '9-16_myxv-r1_16_S33',
 '9-20_myxv-r1_20_S45',
 '10-0_myxv-r2_0_S10',
 '10-12_myxv-r2_12_S22',
 '10-16_myxv-r2_16_S34',
 '10-20_myxv-r2_20_S46',
 '11-0_tpv-r1_0_S11',
 '11-12_tpv-r1_12_S23',
 '11-16_tpv-r1_16_S35',
 '11-20_tpv-r1_20_S47',
 '12-0_tpv-r2_0_S12',
 '12-12_tpv-r2_12_S24',
 '12-16_tpv-r2_16_S36',
 '12-20_tpv-r2_20_S48'
]

In [18]:
experiment = [samples[i:i+4] for i in range(0, len(samples), 4)]
conditions_list = ['K3Δ58', 'K3Δ58', 'RCVZ vIF2α', 'RCVZ vIF2α', 'VACV K3', 'VACV K3', 'VARV C3', 'VARV C3', 'MYXV M156R','MYXV M156R', 'TPV K3', 'TPV K3']
rep_list = [1,2,1,2,1,2,1,2,1,2,1,2]

In [ ]:
# 1 - generate replicate dataframes (all-reads)

input_path = '../../data/bartender'

output_file = f'../../results/barseq/replicates_all-reads_{date}.csv'

# merge timepoints into single df
df_list = []
for samples,cond,rep in zip(experiment, conditions_list,rep_list):
    for sample in samples:
        #sample = sample.rsplit('_', 1)[0]
        temp_df = pd.read_csv(f'{input_path}/{sample}.pkr_barcode.txt', names=['pkr_bc', 'line'])
        temp_df = temp_df['pkr_bc'].value_counts().rename_axis('pkr_bc').reset_index(name=sample)
        if sample == samples[0]:
            r_df = temp_df
        else:
            r_df = pd.merge(r_df, temp_df, on='pkr_bc',how='outer')

    # now cleanup r_df before appending to list
    new_cols = ['pkr_bc','0hr','12hr','16hr','20hr']
    r_df.columns = new_cols

    # replicate
    r_df['replicate'] = f'Replicate {rep}'
    
    # k3l variant
    k3_value = f"K3L-{samples[0].split('_')[0]}"
    r_df['k3'] = cond

    # map pkr bc to variant
    r_df['pkr'] = r_df['pkr_bc'].map(pkr_dict)

    # calculations: normalize reads, -log2(fold change), and auc
    r_df = timepoint_calculations(r_df)

    df_list.append(r_df)

# merge all the replicate dataframes
df = pd.concat(df_list, ignore_index=True)

# add metadata
df = pkr_metadata(df)

In [21]:
# save dataframe
df.to_csv(output_file, index=False)

In [11]:
# 2 - combine reads (all-reads and variant-reads)
# input/output
#input_file = f'../../results/barseq/replicates_all-reads_{date}.csv'
input_file = '../../results/barseq/replicates_all-reads_241011.csv'

output_all_reads = f'../../results/barseq/combined_all-reads_{date}.csv'
output_file = f'../../results/barseq/combined_variant-reads_{date}.csv'

df = pd.read_csv(input_file)

/tmp/ipykernel_2270431/1714157571.py:9: DtypeWarning: Columns (7,18,19,20,21,22,23,24,25,26,27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


In [12]:
# merge all the replicate dataframes
grp_cols = ['k3','pkr','pkr_bc']
cols = ['0hr','12hr','16hr','20hr']
df = df.groupby(grp_cols)[cols].sum().reset_index()

In [13]:
df.k3.unique().tolist()

['K3Δ58', 'MYXV M156R', 'RCVZ vIF2α', 'TPV K3', 'VACV K3', 'VARV C3']

In [14]:
# split datafrmae into k3s and make calculations, then concat
cond_list = ['K3Δ58', 'RCVZ vIF2α', 'VACV K3', 'VARV C3','MYXV M156R','TPV K3']

df_list = []
for cond in cond_list:
    r_df = df.query('k3 == @cond')

    # map pkr bc to variant
    r_df['pkr'] = r_df['pkr_bc'].map(pkr_dict)

    # normalize reads, -log2(fold change), and auc
    r_df = timepoint_calculations(r_df)

    df_list.append(r_df)

df = pd.concat(df_list)

/tmp/ipykernel_2270431/1216727249.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r_df['pkr'] = r_df['pkr_bc'].map(pkr_dict)
/gpfs/gsfs10/users/chambersmj/sadhu_lab/data/experiments/241010_nextseq_k3-orthologs/workflow/nb/timepoint_calculations_v2.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[timepoints] = df[timepoints].fillna(0)
/gpfs/gsfs10/users/chambersmj/sadhu_lab/data/experiments/241010_nextseq_k3-orthologs/workflow/nb/timepoint_calculations_v2.py:14: SettingWithCopyWarning: 
A value is 

In [15]:
# save all reads with unidentified
df.to_csv(output_all_reads, index=False)

In [16]:
# drop missing pkr barcodes
df.dropna(subset=['pkr'], inplace=True)

# apply read threshold
threshold = 50
df = df[df['0hr'] >= threshold]

# add metadata
df = pkr_metadata(df)

# purify to designed pkr variants
input_file = '../../data/dms_primers/pkr_variants_list.pkl'
with open(input_file, 'rb') as f:
    designed_variant_list = pickle.load(f)
designed_variant_list = ["PKR-" + variant for variant in designed_variant_list]
designed_variant_list.append("PKR-WT")
df = df[df['pkr'].isin(designed_variant_list)]

# save dataframe
df.to_csv(output_file, index=False)

/gpfs/gsfs10/users/chambersmj/sadhu_lab/data/experiments/241010_nextseq_k3-orthologs/workflow/nb/add_pkr_metadata.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pkr_type'] = df['pkr'].apply(pkr_type)
/gpfs/gsfs10/users/chambersmj/sadhu_lab/data/experiments/241010_nextseq_k3-orthologs/workflow/nb/add_pkr_metadata.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pkr'].replace(np.nan, '', inplace=True)
/gpfs/gsfs10/users/chambersmj/sadhu_lab/data/experiments/241010_nextseq_k3-orthologs/workflow/nb/add_pkr_metadata.py:60: SettingWithCopyWarning: 
A 

In [17]:
# 3 - groupby pkr variants and condense the dataframe by replicate
old_date = '241011'
input_file = f'../../results/barseq/combined_variant-reads_{date}.csv'
output_file = f'../../results/barseq/combined_grouped-barcodes_{date}.csv'

df = pd.read_csv(input_file)

result_df = df.groupby(['k3', 'pkr']).agg({
    'auc': ['mean', 'std', 'sem'],
    'pkr_bc': 'nunique',
    '0hr': 'sum'
}).reset_index()

result_df.columns = ['k3', 'pkr', 'auc_mean', 'auc_std', 'auc_sem', 'pkr_bc_nunique', '0hr_total_reads']

# purify to designed pkr variants
'''
input_file = '../../data/dms_primers/pkr_variants_list.pkl'
with open(input_file, 'rb') as f:
    designed_variant_list = pickle.load(f)
designed_variant_list = ["PKR-" + variant for variant in designed_variant_list]
designed_variant_list.append("PKR-WT")
df = df[df['pkr'].isin(designed_variant_list)]
'''

df = pkr_metadata(result_df)

df.to_csv(output_file, index=False)

In [18]:
df

,k3,pkr,auc_mean,auc_std,auc_sem,pkr_bc_nunique,0hr_total_reads,site,pkr_regions,pkr_type,...,wt_cat,var_cat,isr_kinase_conservation,elde_primate_positive_selection,vertebrate_positive_selection,pkr_conservation,jaquet_primate_positive_selection,vert_sele_analysis,elde_primate_sele_analysis,jaquet_primate_sele_analysis
0,K3Δ58,PKR-A488D,21.698643,6.686489,1.854498,13,3187.0,488.0,Region 4,Variant,...,Non-Polar,Negative Charge,NaN,NaN,Vertebrate Positive Selection,NaN,NaN,Vertebrate Positive Selection,NaN,NaN
1,K3Δ58,PKR-A488G,23.940283,7.777414,2.078601,14,3970.0,488.0,Region 4,Variant,...,Non-Polar,Unique,NaN,NaN,Vertebrate Positive Selection,NaN,NaN,Vertebrate Positive Selection,NaN,NaN
2,K3Δ58,PKR-A488P,-34.590434,1.751955,0.583985,9,3073.0,488.0,Region 4,Variant,...,Non-Polar,Unique,NaN,NaN,Vertebrate Positive Selection,NaN,NaN,Vertebrate Positive Selection,NaN,NaN
3,K3Δ58,PKR-A488S,20.209814,5.898142,1.353127,19,4567.0,488.0,Region 4,Variant,...,Non-Polar,Polar-Neutral,NaN,NaN,Vertebrate Positive Selection,NaN,NaN,Vertebrate Positive Selection,NaN,NaN
4,K3Δ58,PKR-A488T,20.711894,8.937464,2.307643,15,3644.0,488.0,Region 4,Variant,...,Non-Polar,Polar-Neutral,NaN,NaN,Vertebrate Positive Selection,NaN,NaN,Vertebrate Positive Selection,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2581,VARV C3,PKR-Y454D,-16.338699,3.590470,0.615760,34,6573.0,454.0,Region 3,Variant,...,Aromatic,Negative Charge,Conserved Kinase Residue,NaN,NaN,Conserved PKR Residue,NaN,Conserved PKR Residue,Conserved PKR Residue,Conserved PKR Residue
2582,VARV C3,PKR-Y454F,15.883250,7.477308,1.413078,28,5726.0,454.0,Region 3,Variant,...,Aromatic,Aromatic,Conserved Kinase Residue,NaN,NaN,Conserved PKR Residue,NaN,Conserved PKR Residue,Conserved PKR Residue,Conserved PKR Residue
2583,VARV C3,PKR-Y454H,14.402763,7.293133,1.354301,29,5427.0,454.0,Region 3,Variant,...,Aromatic,Positive Charge,Conserved Kinase Residue,NaN,NaN,Conserved PKR Residue,NaN,Conserved PKR Residue,Conserved PKR Residue,Conserved PKR Residue
2584,VARV C3,PKR-Y454N,-10.880635,3.960787,0.669495,35,7477.0,454.0,Region 3,Variant,...,Aromatic,Polar-Neutral,Conserved Kinase Residue,NaN,NaN,Conserved PKR Residue,NaN,Conserved PKR Residue,Conserved PKR Residue,Conserved PKR Residue
